In [0]:
display(dbutils.fs.ls("/Volumes/workspace/bd26049/datos"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/bd26049/datos/DataCoSupplyChainDataset.csv,DataCoSupplyChainDataset.csv,95910149,1795583571000
dbfs:/Volumes/workspace/bd26049/datos/WhatsApp Image 2026-11-26 at 12.14.17 AM.jpeg,WhatsApp Image 2026-11-26 at 12.14.17 AM.jpeg,229000,1795670843000


# Experimento de Optimización en Apache Spark (DataCo)
Comparación de tiempos de ejecución entre un flujo base sin optimizar y un flujo optimizado (proyección de columnas, renombrado sin espacios, particionado y caché) sobre `DataCoSupplyChainDataset.csv`.

In [0]:
import time
from pyspark.sql import functions as F

ruta_csv = "/Volumes/workspace/bd26049/datos/DataCoSupplyChainDataset.csv"

t0 = time.time()

dfBase = (spark.read
  .format("csv")
  .option("header", "true")
  .option("inferSchema", "true")
  .load(ruta_csv)
)

resBase = (dfBase
  .groupBy("Category Name", "Shipping Mode")
  .agg(
      F.count("*").alias("total_registros"),
      F.sum("Sales").alias("ventas_totales")
  )
  .orderBy(F.desc("ventas_totales"))
)

resBase.collect()
tiempo_base = round(time.time() - t0, 4)
print("Tiempo Escenario 1 (Sin optimizar):", tiempo_base, "s")

Tiempo Escenario 1 (Sin optimizar): 2.2507 s


## Optimización: proyección de columnas, renombrado, particionado y caché
Se seleccionan únicamente las columnas requeridas, se renombran sin espacios para evitar conflictos y se aplica `repartition` y `cache()`.

In [0]:
t1 = time.time()

dfOpt = (dfBase
  .select(
      F.col("Category Name").alias("categoria"),
      F.col("Shipping Mode").alias("modo_envio"),
      F.col("Sales").cast("double").alias("ventas")
  )
  .filter("ventas > 0")
  .repartition(4, "categoria")
  .cache()
)

dfOpt.count()  # Dispara la materialización en caché

resOpt = (dfOpt
  .groupBy("categoria", "modo_envio")
  .agg(
      F.count("*").alias("total_registros"),
      F.sum("ventas").alias("ventas_totales")
  )
  .orderBy(F.desc("ventas_totales"))
)

resOpt.collect()
tiempo_opt = round(time.time() - t1, 4)
print("Tiempo Escenario 2 (Optimizado - 1ra corrida):", tiempo_opt, "s")

Tiempo Escenario 2 (Optimizado - 1ra corrida): 5.3098 s


In [0]:
t2 = time.time()
resOpt.collect()
tiempo_cache = round(time.time() - t2, 4)

print("Tiempo Escenario 2B (Consulta desde caché):", tiempo_cache, "s")
print(f"Resumen -> Base: {tiempo_base}s | Optimizado (1ra): {tiempo_opt}s | Caché: {tiempo_cache}s")
display(resOpt.limit(10))

Tiempo Escenario 2B (Consulta desde caché): 0.6511 s
Resumen -> Base: 2.2507s | Optimizado (1ra): 5.3098s | Caché: 0.6511s


categoria,modo_envio,total_registros,ventas_totales
Fishing,Standard Class,10428,4170991.554707242
Cleats,Standard Class,14637,2647359.6534935995
Camping & Hiking,Standard Class,8266,2479634.7709259344
Cardio Equipment,Standard Class,7497,2211250.5274535096
Women's Apparel,Standard Class,12526,1869900.0
Water Sports,Standard Class,9177,1838758.280121326
Indoor/Outdoor Games,Standard Class,11534,1728258.4038768413
Men's Footwear,Standard Class,13288,1727307.1930836479
Fishing,Second Class,3314,1325533.7564539858
Fishing,First Class,2715,1085945.7298649468


Databricks visualization. Run in Databricks to view.

## Conclusión del experimento en Databricks
- La primera ejecución optimizada (`5.3098 s`) incurre en el costo inicial de redistribuir particiones (`repartition`) y materializar el DataFrame en caché (`cache()`).
- En consultas posteriores sobre el DataFrame cacheado (`0.6511 s`), el tiempo de ejecución se reduce drásticamente frente al escenario base sin optimizar (`2.2507 s`), evidenciando el beneficio de `cache()` para cargas analíticas recurrentes.